# Huấn luyện YOLOv8 · lượt so sánh chính và AI-07 · Signa v1.1, v1.1-syn

Giao thức [Thiết kế AI §3](https://github.com/hana200804/DA/blob/f4ad6edc24d26f0cb6d937c8331ba48f98216234/docs/04-thiet-ke-ai.md#3-giao-thức-huấn-luyện):
khởi tạo COCO, tối đa 200 epoch, dừng sớm khi mAP@0.5:0.95 trên val không tăng sau 30 epoch, seed 0.
Tăng cường theo [bảng chung](https://github.com/hana200804/DA/blob/f4ad6edc24d26f0cb6d937c8331ba48f98216234/ai/configs/augment.md).
Kích thước ảnh, cỡ mô hình, batch và bộ dữ liệu đọc từ biến môi trường: `SIGNA_IMGSZ` (mặc định 640), `SIGNA_SIZES` (mặc định `n,s`), `SIGNA_BATCH` (mặc định 16 như lượt chính), `SIGNA_DATASET_VERSION` (mặc định `v1.1`, nhận `v1.1` hoặc `v1.1-syn`).

1. Runtime → Change runtime type → **L4 GPU** (hoặc T4).
2. `signa-v1.1.zip` nằm ở thư mục gốc My Drive.
3. Lấy SHA từ liên kết cố định GitHub của bản notebook đang chạy (`…/blob/<SHA>/…`). Thêm một ô mã ở đầu notebook trên Colab với `%env SIGNA_NOTEBOOK_COMMIT=<SHA>` và thay `<SHA>` bằng giá trị đó.
4. **Runtime → Run all.** Notebook train lần lượt từng cỡ trong `SIZES`, xuất dự đoán val và chấm bằng bộ đánh giá chung.
5. Mất phiên: mở lại, **Run all** lần nữa. Run đã xong được bỏ qua, run dở tiếp tục từ `last.pt` với trạng thái dừng sớm cùng epoch.
6. Giữ nguyên các tham số khi resume. Để chủ động đổi batch, đặt `SIGNA_BATCH` và `SIGNA_ALLOW_BATCH_CHANGE=1`. Mỗi phiên ghi batch trong `meta.json`. Checkpoint cũ thiếu `early-stopping.json` cần tạo lượt mới để giữ đúng patience.

Kết quả ở `My Drive/Signa/experiments/yolov8{size}-{IMGSZ}-{SIGNA_DATASET_VERSION}-main-s0/`: `config.yaml`, `metrics.json`, `best-selection.json`, `meta.json`, `results.csv`, trọng số.
Lượt chính 640 cho ra `yolov8n-640-v1.1-main-s0` và `yolov8s-640-v1.1-main-s0`.
Không xuất test trong notebook này. Test chỉ chạy sau khi chốt cấu hình (docs/04 §4).

### Chạy AI-07 · YOLOv8s ảnh 960

1. Làm bước 2 và 3 ở trên. Trong ô mã thêm ở đầu, cạnh `%env SIGNA_NOTEBOOK_COMMIT=<SHA>`, thêm hai dòng `%env SIGNA_IMGSZ=960` và `%env SIGNA_SIZES=s`. Không sửa ô mã nào khác.
2. Chọn **L4 GPU** (hoặc A100). Không dùng T4 cho lượt này.
3. **Runtime → Run all**. Mất phiên thì giữ ô `%env` như trên rồi **Run all**, run dở tiếp tục từ `last.pt` với trạng thái dừng sớm cùng epoch.

Kết quả ở `My Drive/Signa/experiments/yolov8s-960-v1.1-main-s0/`.
Ước lượng, chưa đo: lượt YOLOv8s 640 mất 6,1 giờ L4; ảnh 960 có khoảng 2,25 lần số điểm ảnh, nếu thời gian tăng theo số điểm ảnh thì lượt này cần khoảng 13,7 giờ L4.
Batch mặc định là 16 như lượt chính. Khi hết bộ nhớ GPU, đặt `%env SIGNA_BATCH=<số nhỏ hơn>`. Nếu đã có checkpoint, đặt thêm `%env SIGNA_ALLOW_BATCH_CHANGE=1` cho phiên đổi batch. Báo cáo batch theo phiên từ `meta.json`.

### Chạy AI-07 · có/không dữ liệu tổng hợp v1.1-syn

Lượt đối chứng là `yolov8n-640-v1.1-main-s0` đã có. Lượt mới chỉ đổi bộ dữ liệu: cùng YOLOv8n, ảnh 640, batch 16, seed 0, cùng bảng tăng cường.

1. Tải `signa-v1.1-syn.zip` lên thư mục gốc My Drive, cạnh `signa-v1.1.zip`. Thư mục gốc trong ZIP là `v1.1-syn/`.
2. Trong ô mã thêm ở đầu, cạnh `%env SIGNA_NOTEBOOK_COMMIT=<SHA>`, thêm ba dòng `%env SIGNA_DATASET_VERSION=v1.1-syn`, `%env SIGNA_SIZES=n` và `%env SIGNA_IMGSZ=640`. Không đặt `SIGNA_BATCH`. Không sửa ô mã nào khác.
3. Chọn **L4 GPU**, rồi **Runtime → Run all**. Mất phiên thì giữ ô `%env` như trên rồi **Run all**.

Notebook đối chiếu `manifest.csv` trong gói với `ai/data/manifests/v1.1-syn.csv`, kiểm các dòng val, test, field trùng hệt `v1.1.csv`, mọi ảnh `source=synthetic` thuộc train và `coco/val.json` trùng SHA-256 của v1.1.
Kết quả ở `My Drive/Signa/experiments/yolov8n-640-v1.1-syn-main-s0/`. `config.yaml`, `meta.json`, `metrics.json` và `evaluation/export-config.json` ghi `v1.1-syn`.
Ước lượng, chưa đo: lượt gốc mất 5,48 giờ L4 với 6.679 ảnh train. v1.1-syn có 7.297 ảnh train, nhiều hơn 9%, nên lượt này cần khoảng 5,5–6 giờ L4 tùy epoch dừng sớm.
So sánh trên **val** bằng `scripts/compare_synthetic.py`. Không dùng test để quyết định có dùng dữ liệu tổng hợp hay không. Xem `ai/README.md`.

## 1. Cài đặt

In [ ]:
import json
import os
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("SIGNA_SMOKE") == "1"  # 1 epoch, 2% train, ghi ra /content
# Lượt chính: mặc định 640, n và s. AI-07 biển nhỏ: %env SIGNA_IMGSZ=960 và %env SIGNA_SIZES=s.
IMGSZ = int(os.environ.get("SIGNA_IMGSZ", "640"))
SIZES = [s.strip() for s in os.environ.get("SIGNA_SIZES", "n,s").split(",")]
assert IMGSZ % 32 == 0 and set(SIZES) <= {"n", "s"}, "IMGSZ chia hết cho 32; SIZES gồm n, s."
# AI-07 có/không dữ liệu tổng hợp: %env SIGNA_DATASET_VERSION=v1.1-syn.
DATASET_VERSION = os.environ.get("SIGNA_DATASET_VERSION", "v1.1")
assert DATASET_VERSION in {"v1.1", "v1.1-syn"}, "SIGNA_DATASET_VERSION là v1.1 hoặc v1.1-syn."
# Đặt SIGNA_NOTEBOOK_COMMIT theo SHA trên permalink GitHub của notebook đang chạy.
NOTEBOOK_COMMIT = os.environ.get("SIGNA_NOTEBOOK_COMMIT") or None
REPO = Path(os.environ.get("SIGNA_REPO", "/content/signa-source"))


def sh(*args):
    subprocess.run([str(a) for a in args], check=True)


if IN_COLAB:
    if NOTEBOOK_COMMIT is None:
        raise ValueError("Đặt SIGNA_NOTEBOOK_COMMIT theo liên kết cố định của notebook.")
    packages = ["ultralytics==8.3.203", "albumentations==2.0.8", "pycocotools"]
    sh(sys.executable, "-m", "pip", "install", "-q", *packages)
    if not (REPO / ".git").exists():
        sh("git", "clone", "-q", "https://github.com/hana200804/DA.git", REPO)
    sh("git", "-C", REPO, "fetch", "-q", "origin")
    sh("git", "-C", REPO, "checkout", "-q", "--detach", NOTEBOOK_COMMIT)
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
sys.path.insert(0, str(REPO / "ai/src"))

## 2. Dữ liệu

In [ ]:
import csv
import hashlib
from zipfile import ZipFile

import torch
import yaml

from signa_ai.dataset import read_yolo, relative_path
from signa_ai.detection import EVAL_CONFIDENCE, Box, Detection, to_coco_predictions
from signa_ai.labels import load_label_map
from signa_ai.yolo_training import (
    check_resume_config,
    file_sha256,
    freeze_requirements,
    load_stop_state,
    restore_stop_state,
    save_stop_state,
    write_run_file,
)

VAL_GT_SHA256 = "e3ccd276014eadd148030941675efeaf06db1c1ad8deddcd80747d40ec8b08e2"  # v1.1


def check_manifest(manifest, manifests, version):
    """Manifest trong gói trùng bản trong Git; v1.1-syn giữ nguyên val, test, field của v1.1."""
    expected = manifests / f"{version}.csv"
    assert manifest.read_bytes() == expected.read_bytes(), f"manifest.csv lệch {expected}"
    if version == "v1.1":
        return

    def read(path):
        with path.open(encoding="utf-8", newline="") as handle:
            return list(csv.DictReader(handle))

    rows, base = read(manifest), read(manifests / "v1.1.csv")
    held_out = {"val", "test", "field"}
    assert [row for row in rows if row["split"] in held_out] == [
        row for row in base if row["split"] in held_out
    ], f"val, test, field của {version} phải trùng hệt v1.1.csv."
    synthetic = [row for row in rows if row["source"] == "synthetic"]
    leaked = [row["path"] for row in synthetic if row["split"] != "train"]
    assert synthetic and not leaked, f"Ảnh tổng hợp phải thuộc train: {leaked[:5]}"


assert torch.cuda.is_available(), "Chọn GPU (L4 hoặc T4) trong Runtime → Change runtime type."
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU)

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    ARCHIVE = Path(f"/content/drive/MyDrive/signa-{DATASET_VERSION}.zip")
    DATASET = Path(f"/content/{DATASET_VERSION}")
    # Smoke ghi ra /content để không lẫn với run thật trên Drive.
    DRIVE_ROOT = Path("/content/drive/MyDrive/Signa/experiments")
    if SMOKE:
        DRIVE_ROOT = Path("/content/signa-smoke")
    assert ARCHIVE.is_file(), f"Thiếu {ARCHIVE}"
    if not DATASET.exists():
        with ZipFile(ARCHIVE) as archive:
            for member in archive.namelist():
                root = relative_path(member).parts[0]
                assert root == DATASET_VERSION, f"ZIP sai thư mục gốc: {member}"
            archive.extractall("/content")
else:
    DATASET = Path(os.environ["SIGNA_DATASET"])
    DRIVE_ROOT = Path(os.environ["SIGNA_RUNS"])

label_map = load_label_map(REPO / "ai/data/labels.yaml")
assert label_map.version == 1 and len(label_map) == 46
NAMES = label_map.names
config_path = Path("/content/signa-data.yaml") if IN_COLAB else DRIVE_ROOT / "data.yaml"
config = yaml.safe_load((DATASET / "data.yaml").read_text())
assert config["names"] == NAMES, "data.yaml lệch danh mục v1."
config["path"] = str(DATASET)
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
check_manifest(DATASET / "manifest.csv", REPO / "ai/data/manifests", DATASET_VERSION)
if DATASET_VERSION != "v1.1":
    assert file_sha256(DATASET / "coco/val.json") == VAL_GT_SHA256, "GT val khác v1.1."

EXPECTED = {"train": (6679, 13283), "val": (1599, 2685), "test": (1700, 2664)}
if DATASET_VERSION == "v1.1-syn":
    # Thêm 618 ảnh tổng hợp: 618 box dán và 422 box thật chép từ ảnh nền.
    EXPECTED["train"] = (7297, 14323)
GROUND_TRUTH = {}
for split, (image_count, box_count) in EXPECTED.items():
    assert config[split] == f"images/{split}"
    truth = json.loads((DATASET / f"coco/{split}.json").read_text())
    assert truth["info"]["label_map_version"] == label_map.version
    assert {c["id"]: c["name"] for c in truth["categories"]} == NAMES
    assert len(truth["images"]) == image_count and len(truth["annotations"]) == box_count
    boxes = 0
    for image in truth["images"]:
        rel = relative_path(image["file_name"])
        assert rel.parts[:2] == ("images", split) and (DATASET / rel).is_file(), str(rel)
        label = DATASET / "labels" / split / rel.with_suffix(".txt").name
        boxes += len(read_yolo(label, label_map))
    assert boxes == box_count
    GROUND_TRUTH[split] = truth
    print(split, image_count, "ảnh,", boxes, "box")

## 3. Cấu hình

In [ ]:
import ultralytics
from ultralytics import YOLO, settings
from ultralytics.utils.metrics import Metric

from signa_ai.augment import ULTRALYTICS_ARGS, patch_ultralytics

assert ultralytics.__version__ == "8.3.203", "Restart runtime rồi chạy lại ô 1."
settings.update({"wandb": False, "mlflow": False, "comet": False, "clearml": False})
os.environ['PYTHONPATH'] = str(REPO / 'ai/src')  # cho subprocess signa_ai.evaluate

# Bản ghim chọn và dừng theo mAP50-95 thuần, không pha mAP50.
probe = Metric()
probe.p, probe.r = torch.ones(1).numpy(), torch.ones(1).numpy()
probe.all_ap = torch.tensor([[0.9] + [0.1] * 9]).numpy()
assert abs(probe.fitness() - probe.map) < 1e-6

patch_ultralytics()  # pipeline Albumentations chung thay pipeline mặc định

SEED = 0
EPOCHS = 1 if SMOKE else 200  # trần YOLO, dừng sớm theo patience
MAIN_BATCH = 16  # batch của lượt chính 640
# Hết bộ nhớ GPU thì giảm bằng %env SIGNA_BATCH=<số>; config.yaml ghi batch_note.
BATCH = int(os.environ.get("SIGNA_BATCH", str(MAIN_BATCH)))
assert BATCH > 0, "SIGNA_BATCH phải là số nguyên dương."
ALLOW_BATCH_CHANGE = os.environ.get("SIGNA_ALLOW_BATCH_CHANGE") == "1"


def run_name(size):
    return f"yolov8{size}-{IMGSZ}-{DATASET_VERSION}-main-s{SEED}"


def train_args(size):
    return dict(
        data=str(config_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        patience=30,
        seed=SEED,
        deterministic=True,
        workers=2,
        cache=False,
        project=str(DRIVE_ROOT),
        name=run_name(size),
        exist_ok=True,
        val=True,
        plots=True,
        amp=True,
        **({"fraction": 0.02} if SMOKE else {}),
        **ULTRALYTICS_ARGS,
    )


def run_config(args):
    """Nội dung config.yaml: phiên bản dữ liệu, nhãn và toàn bộ tham số train."""
    config = {"dataset_version": DATASET_VERSION, "label_map_version": label_map.version, **args}
    if args["batch"] != MAIN_BATCH:
        config["batch_note"] = f"Khác lượt chính: batch {args['batch']} thay vì {MAIN_BATCH}."
    return config


def track(model, run_dir, session_args, stop_state=None, coco_weights_sha256=None):
    meta, session, start = start_session(
        run_dir, {"ultralytics": ultralytics.__version__, **session_args},
        coco_weights_sha256=coco_weights_sha256,
    )

    def on_train_start(trainer):
        if stop_state is not None:
            restore_stop_state(trainer.stopper, stop_state)

        save_model = trainer.save_model

        def save_with_stop_state():
            save_stop_state(run_dir, trainer.stopper, trainer.epoch, epoch_offset=1)
            save_model()

        trainer.save_model = save_with_stop_state

    def update_session(trainer):
        save_session(run_dir, meta, session, start, epoch=trainer.epoch + 1)

    def on_save(trainer):
        update_session(trainer)
        if trainer.fitness == trainer.best_fitness:
            selected = {
                "epoch": trainer.epoch + 1,
                "fitness": trainer.fitness,
                "val": {
                    k: float(v) for k, v in trainer.metrics.items() if k.startswith("metrics/")
                },
            }
            (run_dir / "best-selection.json").write_text(json.dumps(selected, indent=2))
        trainer.plot_metrics()

    model.add_callback("on_train_start", on_train_start)
    model.add_callback("on_model_save", on_save)
    model.add_callback("on_train_end", update_session)


def predict_with(best_model):
    def predict_split(truth):
        predictions = []
        for image in truth["images"]:
            result = best_model.predict(
                str(DATASET / image["file_name"]), imgsz=IMGSZ, conf=EVAL_CONFIDENCE,
                iou=0.7, max_det=300, device=0, verbose=False,
            )[0]
            assert result.orig_shape == (image["height"], image["width"])
            detections = [
                Detection(Box.from_xyxy(*xyxy), NAMES[int(cls)], float(score))
                for xyxy, cls, score in zip(
                    result.boxes.xyxy.cpu().tolist(),
                    result.boxes.cls.cpu().tolist(),
                    result.boxes.conf.cpu().tolist(),
                    strict=True,
                )
            ]
            predictions.extend(to_coco_predictions(image["id"], detections, label_map))
        return predictions

    return predict_split

In [ ]:
def run_status(run_dir):
    """done: đã đánh giá; trained: train xong, chưa đánh giá; resume: còn optimizer; new."""
    if (run_dir / "done.json").exists():
        return "done"
    last = run_dir / "weights/last.pt"
    if not last.is_file():
        return "new"
    # Chỉ nạp checkpoint do nhóm tạo; checkpoint PyTorch có thể thực thi mã khi nạp.
    ckpt = torch.load(last, map_location="cpu", weights_only=False)
    return "resume" if ckpt.get("optimizer") is not None else "trained"


def start_session(run_dir, extra, *, coco_weights_sha256=None):
    meta_path = run_dir / "meta.json"
    notebook = {"notebook_commit": NOTEBOOK_COMMIT}
    if NOTEBOOK_COMMIT is None:
        notebook["notebook_commit_note"] = "Chưa cung cấp SHA qua SIGNA_NOTEBOOK_COMMIT."
    meta = (
        json.loads(meta_path.read_text())
        if meta_path.exists()
        else {
            "repo_commit": REPO_COMMIT,
            **notebook,
            "dataset_version": DATASET_VERSION,
            "label_map_version": label_map.version,
            "imgsz": IMGSZ,
            "augmentation": "signa_ai.augment (configs/augment.md)",
            "sessions": [],
        }
    )
    if coco_weights_sha256 is not None:
        meta.update(repo_commit=REPO_COMMIT, coco_weights_sha256=coco_weights_sha256, **notebook)
        if NOTEBOOK_COMMIT is not None:
            meta.pop("notebook_commit_note", None)
    session = {
        "started_at": datetime.now(UTC).isoformat(),
        "gpu": GPU,
        "torch": torch.__version__,
        "gpu_hours": 0.0,
        **notebook,
        **extra,
    }
    meta["sessions"].append(session)
    write_run_file(meta_path, json.dumps(meta, ensure_ascii=False, indent=2))
    return meta, session, time.monotonic()


def save_session(run_dir, meta, session, start, **fields):
    session["gpu_hours"] = (time.monotonic() - start) / 3600
    session.update(fields)
    write_run_file(run_dir / "meta.json", json.dumps(meta, ensure_ascii=False, indent=2))


def write_predictions_and_evaluate(run_dir, best_path, predict_split, export_test=False):
    """Xuất COCO JSON (conf 0,001) và chấm bằng bộ đánh giá chung. Test chỉ khi đã chốt."""
    evaluation = run_dir / "evaluation"
    evaluation.mkdir(exist_ok=True)
    for split in ("val", "test") if export_test else ("val",):
        predictions = predict_split(GROUND_TRUTH[split])
        output = evaluation / f"predictions-{split}.json"
        output.write_text(json.dumps(predictions, ensure_ascii=False, allow_nan=False))
        print(split, len(predictions), "dự đoán →", output)
    (evaluation / "export-config.json").write_text(
        json.dumps(
            {
                "datasetVersion": DATASET_VERSION,
                "labelMapVersion": label_map.version,
                "weights_sha256": hashlib.sha256(best_path.read_bytes()).hexdigest(),
                "conf": EVAL_CONFIDENCE,
                "iou": 0.7,
                "max_det": 300,
                "imgsz": IMGSZ,
                "test_exported": export_test,
            },
            indent=2,
        )
    )
    sh(
        sys.executable, "-m", "signa_ai.evaluate",
        "--gt", DATASET / "coco/val.json",
        "--pred", evaluation / "predictions-val.json",
        "--manifest", DATASET / "manifest.csv",
        "--labels", REPO / "ai/data/labels.yaml",
        "--out", run_dir / "metrics.json",
    )
    metrics_path = run_dir / "metrics.json"
    metrics = {"dataset_version": DATASET_VERSION, **json.loads(metrics_path.read_text())}
    text = json.dumps(metrics, ensure_ascii=False, allow_nan=False, indent=2)
    metrics_path.write_text(text + "\n")
    summary = {k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}
    print(json.dumps(summary, indent=2))

## 4. Huấn luyện, xuất dự đoán val, chấm điểm

In [ ]:
def train_run(run_dir, size, status):
    args = train_args(size)
    current = run_config(args)
    stop_state = None
    coco_weights_sha256 = None
    session_args = {}
    if status == "resume":
        model = YOLO(str(run_dir / "weights/last.pt"))
        checkpoint = model.ckpt
        saved = dict(checkpoint["train_args"])
        if "device" in saved:
            saved["device"] = str(saved["device"])
        check = {**current, "device": str(args["device"]), "fraction": args.get("fraction", 1.0)}
        batch, previous_batch = check_resume_config(
            run_dir, check, saved, "batch", ALLOW_BATCH_CHANGE
        )
        epoch = checkpoint["epoch"]
        stop_state = load_stop_state(run_dir, epoch, epoch_offset=1)
        if (
            epoch + 1 >= args["epochs"]
            or stop_state["epochs_without_improvement"] >= args["patience"]
        ):
            return
        session_args["checkpoint_batch"] = previous_batch
        train_kwargs = {"resume": True, "batch": batch}
    else:
        model = YOLO(f"yolov8{size}.pt")
        coco_weights_sha256 = file_sha256(Path(model.ckpt_path))
        batch = args["batch"]
        train_kwargs = args

    write_run_file(run_dir / "config.yaml", yaml.safe_dump(current, sort_keys=False))
    session_args.update(batch=batch, requirements_file=freeze_requirements(run_dir))
    track(model, run_dir, session_args, stop_state, coco_weights_sha256)
    model.train(**train_kwargs)


for size in SIZES:
    run_dir = DRIVE_ROOT / run_name(size)
    status = run_status(run_dir)
    print(f"== {run_name(size)}: {status}")
    if status in ("new", "resume"):
        run_dir.mkdir(parents=True, exist_ok=True)
        train_run(run_dir, size, status)
    if status != "done":
        best_path = run_dir / "weights/best.pt"
        best_model = YOLO(str(best_path))
        assert best_model.names == NAMES
        write_predictions_and_evaluate(run_dir, best_path, predict_with(best_model))
        finished = {"finished_at": datetime.now(UTC).isoformat()}
        (run_dir / "done.json").write_text(json.dumps(finished))
        del best_model
        torch.cuda.empty_cache()
print("Xong. Kết quả trên Drive:", DRIVE_ROOT)